## Examination Code

Currently refactoring for better performance.


In [ ]:
import json
import re
import ast
import string
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)

BENCHMARK_DIR = Path("/g/data/sz65/sg9944/temporal-knowledge/data/benchmark_results/")
RUN_TAG = "run_04"
MIN_DATASET_ROWS = 100_000
# Per-question registry (IDs + difficulty) written after section 6
QUESTION_REGISTRY_PATH = BENCHMARK_DIR.parent / f"question_registry_{RUN_TAG}"

# Discrete color palette for models (14+ models) — consistent across explicit, relative, invariant plots
MODEL_COLORS = [
    "#4477AA", # Blue (Strong)
    "#EE6677", # Red/Rose (Vibrant)
    "#228833", # Green (Deep)
    "#CCBB44", # Yellow/Olive
    "#66CCEE", # Cyan (Light)
    "#AA3377", # Purple/Magenta
    "#000000", # Black (High Contrast Anchor)
    "#E69F00", # Orange (Okabe-Ito)
    "#56B4E9", # Sky Blue
    "#009E73", # Bluish Green
    "#F0E442", # Bright Yellow
    "#0072B2", # Dark Blue
    "#D55E00", # Vermillion
    "#CC79A7", # Reddish Purple
    "#332288", # Indigo (Very Dark)
    "#999933", # Olive (Mid)
    "#EEEEEE", # Very Light Grey (Use for background/baseline)
    "#BBBBBB", # Grey (Neutral)
]
# Prompt type colors for bar charts (Invariant, Relative, Explicit) — consistent across EM, F1, Contains
PROMPT_TYPE_COLORS = ["#2166AC", "#B2182B", "#1B9E77"]  # blue, red, teal


def get_model_color_map(model_list):
    """Return dict mapping each model to a distinct color. Use for consistent coloring across plots."""
    ordered = sorted(model_list)
    return {m: MODEL_COLORS[i % len(MODEL_COLORS)] for i, m in enumerate(ordered)}


def strip_2512_bf16(name):
    """Remove -2512-BF16 suffix from model names (Mistral/Ministral)."""
    return re.sub(r"-2512-BF16\s*", "", name, flags=re.IGNORECASE).rstrip("- ")


print("Setup complete.")

In [ ]:
import pandas as pd

df_chronos = pd.read_parquet('./data/final/ChronosAlign.parquet')
df_chronos_dedup = df_chronos.drop_duplicates(subset=['question'])

In [ ]:
# Top and bottom subjects by category (top 10 and bottom 10 each)

def _normalize_to_list(value):
    """Normalize scalar/list-like cells into a list of clean strings."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, (list, tuple, set, np.ndarray)):
        raw_items = list(value)
    else:
        s = str(value).strip()
        if not s:
            return []
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                raw_items = parsed if isinstance(parsed, list) else [s]
            except (ValueError, SyntaxError):
                raw_items = [s]
        else:
            raw_items = [s]

    clean_items = []
    for item in raw_items:
        text = str(item).strip().strip("\"'")
        if text:
            clean_items.append(text)
    return clean_items

df_cat_title = df_chronos_dedup[["categories", "title", "question", "views"]].copy()
df_cat_title["categories"] = df_cat_title["categories"].apply(_normalize_to_list)
df_cat_title["title"] = df_cat_title["title"].apply(_normalize_to_list)

# Filter for rows where pageviews > 0 only
df_cat_title = df_cat_title[df_cat_title["views"] > 0]

# Keep one row per question/category/title combination, with max views for any repeated question/title/category
df_cat_title = df_cat_title.explode("categories").explode("title")
df_cat_title = df_cat_title.dropna(subset=["categories", "title"])
df_cat_title = df_cat_title.drop_duplicates(subset=["question", "categories", "title"])

# For each title/category, sum up views (or use max views per title if preferred), then rank by that
title_category_agg = (
    df_cat_title.groupby(["categories", "title"], as_index=False)
    .agg(
        question_count=("question", "size"),
        total_pageviews=("views", "sum")  # can use "max" if desired: .agg(max_pageviews=("views", "max"))
    )
    .sort_values(["categories", "total_pageviews", "question_count", "title"], ascending=[True, False, False, True])
)

# Top 10 titles for each category
top10_titles_by_category = (
    title_category_agg.groupby("categories", group_keys=False)
    .head(10)
    .assign(rank=lambda d: d.groupby("categories")["total_pageviews"].rank(method="first", ascending=False).astype(int))
    .loc[:, ["categories", "rank", "title", "question_count", "total_pageviews"]]
    .sort_values(["categories", "rank"])
    .reset_index(drop=True)
)

# Bottom 10 titles for each category
# Note: For ties in pageviews, titles earlier alphabetically get shown
bottom10_titles_by_category = (
    title_category_agg.groupby("categories", group_keys=False)
    .tail(10)
    .sort_values(["categories", "total_pageviews", "question_count", "title"], ascending=[True, True, False, True])
    .assign(rank=lambda d: d.groupby("categories").cumcount() + 1)
    .loc[:, ["categories", "rank", "title", "question_count", "total_pageviews"]]
    .sort_values(["categories", "rank"])
    .reset_index(drop=True)
)

print("Top 10 titles for each category (views > 0, ordered by pageviews):")
display(top10_titles_by_category)
print("Bottom 10 titles for each category (views > 0, ordered by pageviews):")
display(bottom10_titles_by_category)

In [ ]:
# Distribution of questions per pageview decile, split by category.
# Top is decile 10 (highest views), bottom is decile 1 (lowest views).

# Compute deciles for the "views" column within each category (so that deciles are relative within each group)
df_cat_title["views_decile"] = (
    df_cat_title.groupby("categories")["views"]
    .transform(lambda x: pd.qcut(x, 10, labels=False, duplicates="drop") + 1)
)

# For each decile and category, get up to 3 example titles
examples_by_cat_decile = (
    df_cat_title.groupby(["categories", "views_decile"])
    .agg(example_titles=("title", lambda x: list(dict.fromkeys(x))[:3]))
    .reset_index()
)

# Pivot to have decile as rows, categories as columns, values as example_titles
# Invert order: Top = 10, Bottom = 1
decile_example_table = (
    examples_by_cat_decile
    .pivot(index="views_decile", columns="categories", values="example_titles")
    .reset_index()
    .sort_values("views_decile", ascending=False)  # Top decile first
)

print("Example titles by pageview decile (top=10, decile as row, category groups as columns):")
display(decile_example_table)

In [ ]:
# Load one benchmark parquet to get the base dataset columns
COLS_BASE = ["question", "answer", "relationship", "categories", "event_year",
             "subject", "eventLabel", "id"]

ref_file = next(BENCHMARK_DIR.glob(f"benchmark_*_{RUN_TAG}.parquet"))
print(f"Loading base data from: {ref_file.name}")
df_base = pd.read_parquet(ref_file, columns=COLS_BASE)
print(f"Loaded {len(df_base):,} rows")
# Stable integer per distinct question text (sorted factorization — reproducible)
_q_codes, _ = pd.factorize(df_base["question"], sort=True)
df_base["question_id"] = _q_codes.astype(np.int32)
print(f"Distinct questions: {df_base['question_id'].max() + 1:,}")

import pyarrow.parquet as pq

def pick_cm_f1_merge_keys(df, parquet_schema_names):
    # Unique key columns present in both df_base and benchmark parquet files.
    singles = ()
    composites = (
        ["question", "event_year"],
    )
    for keys in list(singles) + list(composites):
        if not all(k in df.columns for k in keys):
            continue
        if not all(k in parquet_schema_names for k in keys):
            continue
        if df.duplicated(subset=keys).sum() > 0:
            continue
        return list(keys)
    raise ValueError(
        "Could not pick unique CM/F1 merge keys present in both df_base and benchmark parquet. "
        f"df_base columns: {list(df.columns)}"
    )

_parquet_cols = set(pq.read_schema(ref_file).names)
CM_F1_MERGE_KEYS = pick_cm_f1_merge_keys(df_base, _parquet_cols)
print(f"CM/F1 alignment merge keys: {CM_F1_MERGE_KEYS}")

# Deduplicate by the selected key(s)
n_before = len(df_base)
df_base = df_base.drop_duplicates(subset=CM_F1_MERGE_KEYS).reset_index(drop=True)
n_after = len(df_base)
n_dupes = n_before - n_after
if n_dupes > 0:
    print(f"WARNING: Dropped {n_dupes:,} duplicate rows in df_base using keys: {CM_F1_MERGE_KEYS}")

df_base.head(3)

In [ ]:
NUMERIC_PATTERNS = [
    r"^\d+$",                         # pure integer
    r"^\d+\.\d+$",                    # decimal
    r"^\d{1,2}:\d{2}",                # time (mm:ss or h:mm:ss)
    r"^\d+\s*(seconds?|minutes?|hours?|days?|years?|km|mi|m|kg|lbs?)\b",
    r"^\$[\d,]+",                     # currency
    r"^[\d,]+\s*(million|billion|thousand)",
    r"^\d{1,2}[–\-]\d{1,2}$",        # score like 3-1
    r"^\d{4}$",                       # standalone year
]
NUMERIC_RE = re.compile("|".join(NUMERIC_PATTERNS), re.IGNORECASE)


def _first_answer(val):
    """Extract first element from answer (stored as list/array)."""
    if isinstance(val, (list, np.ndarray)):
        return str(val[0]) if len(val) > 0 else ""
    s = str(val).strip()
    if s.startswith("["):
        try:
            p = ast.literal_eval(s)
            return str(p[0]) if isinstance(p, list) and p else s
        except (ValueError, SyntaxError):
            pass
    return s


def is_numerical_answer(val) -> bool:
    text = _first_answer(val).strip()
    if not text:
        return False
    return bool(NUMERIC_RE.match(text))


df_base["answer_first"] = df_base["answer"].apply(_first_answer)
df_base["answer_type"] = df_base["answer_first"].apply(
    lambda a: "numerical" if is_numerical_answer(a) else "text"
)

type_counts = df_base["answer_type"].value_counts()
print("Answer type distribution:")
print(type_counts)
print(f"\nNumerical: {type_counts.get('numerical', 0):,} ({type_counts.get('numerical', 0)/len(df_base)*100:.1f}%)")
print(f"Text:      {type_counts.get('text', 0):,} ({type_counts.get('text', 0)/len(df_base)*100:.1f}%)")

In [ ]:
# Breakdown by category
ct = pd.crosstab(df_base["categories"], df_base["answer_type"], margins=True)
ct_pct = pd.crosstab(df_base["categories"], df_base["answer_type"], normalize="index") * 100
print("Answer type counts by category:")
display(ct)
print("\nAnswer type % by category:")
display(ct_pct.round(1))

In [ ]:
# Pie chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].pie(type_counts.values, labels=type_counts.index, autopct="%1.1f%%",
            startangle=90, colors=["#4C72B0", "#DD8452"])
axes[0].set_title("Overall Answer Type Distribution")

# By category stacked bar
ct_pct.plot.barh(stacked=True, ax=axes[1], color=["#DD8452", "#4C72B0"])
axes[1].set_xlabel("%")
axes[1].set_title("Answer Type by Category")
axes[1].legend(title="Answer Type", loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
def normalise(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())


def _as_list(val):
    if isinstance(val, (list, np.ndarray)):
        return [str(v) for v in val]
    s = str(val).strip()
    if s.startswith("["):
        try:
            p = ast.literal_eval(s)
            return [str(v) for v in p] if isinstance(p, list) else [s]
        except (ValueError, SyntaxError):
            pass
    return [s]


def exact_match(pred: str, truth) -> float:
    pred = normalise(pred)
    return 1.0 if any(normalise(t) == pred for t in _as_list(truth)) else 0.0


def token_f1(pred: str, truth) -> float:
    pred_tokens = normalise(pred).split()
    truths = _as_list(truth)
    if not pred_tokens:
        return 0.0
    best = 0.0
    for t in truths:
        tt = normalise(t).split()
        if not tt:
            continue
        common = Counter(pred_tokens) & Counter(tt)
        n = sum(common.values())
        if n == 0:
            continue
        prec = n / len(pred_tokens)
        rec = n / len(tt)
        best = max(best, 2 * prec * rec / (prec + rec))
    return best


def contains_match(pred: str, truth) -> float:
    pred = normalise(pred)
    return 1.0 if any(normalise(t) in pred for t in _as_list(truth)) else 0.0


print("Scoring functions defined.")

In [ ]:
# Discover all run_04 vllm parquet files (exclude base-pretrained with small row counts)
parquet_files = sorted(BENCHMARK_DIR.glob(f"benchmark_*_{RUN_TAG}.parquet"))
parquet_files = [f for f in parquet_files if "scored" not in f.name]

ANSWER_COLS = ["answer", "model_answer_invariant", "model_answer_relative", "model_answer_explicit"]

print(f"Found {len(parquet_files)} benchmark parquet files:")
for f in parquet_files:
    print(f"  {f.name} ({f.stat().st_size / 1e6:.0f} MB)")

In [ ]:
summary_files = sorted(BENCHMARK_DIR.glob(f"benchmark_summary_*_{RUN_TAG}.json"))
print(f"Found {len(summary_files)} summary files for run tag '{RUN_TAG}'")

summaries = []
for f in summary_files:
    with open(f) as fh:
        d = json.load(fh)
    summaries.append(d)

df_summary = pd.DataFrame(summaries)

# Filter to consistent runs (same dataset size)
# df_summary = df_summary[df_summary["dataset_rows"] >= MIN_DATASET_ROWS].reset_index(drop=True)
# print(f"Keeping {len(df_summary)} models with >= {MIN_DATASET_ROWS:,} rows")

# Short model names for display (strip -2512-BF16 from Mistral/Ministral)
df_summary["model_short"] = df_summary["model"].apply(
    lambda m: strip_2512_bf16(m.split("/")[-1])
)
df_summary = df_summary.sort_values("explicit_em", ascending=False).reset_index(drop=True)
df_summary[["model_short", "dataset_rows", "invariant_em", "invariant_f1", "invariant_contains",
            "relative_em", "relative_f1", "relative_contains",
            "explicit_em", "explicit_f1", "explicit_contains",
            "total_wall_time_min"]]